# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


openai = OpenAI()

API key looks good so far


In [3]:
# set up environment

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [4]:
# here is the question; type over this to ask something new

#question = """
#Please explain what this code does and why:
#yield from {book.get("author") for book in books if book.get("author")}
#"""

code_snippet = """
yield from {book.get("author") for book in books if book.get("author")}
"""

def create_learning_question(code_snippet):


    system_message = """
    You are a patient, educational coding mentor helping someone learn programming.
    Your goal is to explain code in a way that helps them understand not just what
    the code does, but how to recognize and apply similar patterns in the future.
    
    Always:
    - Break things down step-by-step
    - Explain the "why" behind approaches, not just the "what"
    - Compare to similar patterns when relevant
    - Be clear and thorough
    """



    user_message = f"""
    Please explain this code step-by-step in a way that helps me learn:

    {code_snippet}

    Structure your explanation as:
    1. **High-level overview**: What problem does this solve?
    2. **Step-by-step breakdown**: Walk through each part of the code
    3. **Key concepts**: What programming patterns/techniques are used here?
    4. **Why this approach**: What are the tradeoffs vs alternative approaches?
    5. **Similar patterns**: How does this compare to other approaches I might have seen?
    6. **Practical application**: When would I use similar patterns in my own code?
    7. **Watch out for**: Any potential gotchas or edge cases?

    Make it educational - help me understand how to recognize and apply similar patterns in future code.
    """

    return system_message, user_message

In [ ]:
# Code explanation tool: Unified interface for any OpenAI-compatible model

def ask_about_code(code_snippet, model_client, model_name, stream=False, should_display=True, return_full=False):
    """
    Get explanation of code from any model.
    
    Args:
        code_snippet: Code to explain
        model_client: OpenAI client (configured for OpenAI or Ollama)
        model_name: Model name string (e.g., 'gpt-4o-mini' or 'llama3.2')
        stream: Whether to stream response
        should_display: Whether to display result (notebook-friendly)
        return_full: Return full response object vs just text
    
    Returns:
        Response text (or full response if return_full=True, or dict for streaming+return_full)
    """
    system_msg, user_msg = create_learning_question(code_snippet)
    
    try:
        if stream:
            collected_text = ""
            display_handle = display(Markdown(""), display_id=True) if should_display else None
            
            stream_response = model_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ],
                stream=True
            )
            
            for chunk in stream_response:
                chunk_text = chunk.choices[0].delta.content or ''
                collected_text += chunk_text
                if should_display and display_handle:
                    update_display(Markdown(collected_text), display_id=display_handle.display_id)
            
            if return_full:
                return {
                    "text": collected_text,
                    "model": model_name,
                    "streaming": True
                }
            else:
                return collected_text
        else:
            response = model_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ]
            )
            
            result_text = response.choices[0].message.content
            
            if should_display:
                display(Markdown(result_text))
            
            if return_full:
                return response
            else:
                return result_text
                
    except Exception as e:
        print(f"Error calling {model_name}: {e}")
        return None

def compare_explanations(code_snippet, gpt_client, llama_client):
    """Get explanations from both models and compare"""
    print("="*60)
    print(f"{MODEL_GPT} Explanation:")
    print("="*60)
    gpt_result = ask_about_code(code_snippet, gpt_client, MODEL_GPT, stream=False, should_display=True)
    
    print("\n" + "="*60)
    print(f"{MODEL_LLAMA} Explanation:")
    print("="*60)
    llama_result = ask_about_code(code_snippet, llama_client, MODEL_LLAMA, stream=False, should_display=True)
    
    return gpt_result, llama_result



In [ ]:
# Testing and usage examples

result = ask_about_code(code_snippet, openai, MODEL_GPT, stream=True)


#full_response = ask_about_code(code_snippet, openai, MODEL_GPT, return_full=True)
#print(f"Tokens: {full_response.usage.total_tokens}")


#gpt_result, llama_result = compare_explanations(code_snippet, openai, ollama)

Sure! Let’s break down the provided code step-by-step. 

### 1. **High-level overview**: 
This code is a way to extract unique authors from a collection of book information. It uses a generator to yield each author if they exist, helping to avoid duplicates and improve memory efficiency.

### 2. **Step-by-step breakdown**:
   - **`{book.get("author") for book in books if book.get("author")}`**:
     - This is a set comprehension. A set is a collection that only allows unique items.
     - `for book in books` iterates over each `book` in the `books` collection (which we assume is a list or similar iterable).
     - `book.get("author")` attempts to fetch the `author` value from each `book` dictionary. The `get()` method returns `None` if the key doesn’t exist rather than raising an error.
     - The `if book.get("author")` acts as a filter. It only includes the `author` in the set if the `author` exists (i.e., it’s not `None` or an empty string).
   - **`yield from`**:
     - This is used to yield values from an iterator (in this case, the set created above) one at a time. This means that instead of returning the entire set at once, it produces each unique author one by one when called.

### 3. **Key concepts**:
- **Set comprehensions**: This is a concise way to create a set based on an existing iterable, ensuring uniqueness automatically.
- **Generator functions**: Using `yield` makes this function a generator, which allows for efficient memory usage since it only produces one result at a time.
- **Conditional expressions**: Filtering with the `if` statement in the comprehension is a common practice to ensure you only include valid data.

### 4. **Why this approach**:
- **Tradeoffs**:
   - **Efficiency**: Using a set comprehension combined with `yield from` is efficient in terms of both time (due to the set's unique property) and space (due to the generator yielding results one at a time).
   - **Simplicity**: This line of code is compact and expressive, making it easy to read if you are familiar with Python’s syntax.
   - **Alternative approaches** could include using a loop to create a list of authors and then filtering out duplicates afterwards, but that would involve more memory and complexity.

### 5. **Similar patterns**:
- **List comprehensions**: You might have seen something similar with lists, which could be used if you needed ordered results with possible duplicates (e.g., `[book.get("author") for book in books if book.get("author")]`).
- Using `for` loops instead of comprehensions could be an alternative but would be more verbose and likely less readable.
- Another alternative would be using just `filter()` or `map()` alongside conversion to a list or set, but comprehension is often clearer.

### 6. **Practical application**:
Use similar patterns when you need to:
- Collect unique items from a larger dataset (like authors, product IDs, etc.).
- Process data streams where you can’t hold everything in memory (like reading line-by-line from a file).
- Create generators for processing large datasets one piece at a time, which improves performance for long data processing tasks.

### 7. **Watch out for**: 
- **Missing keys**: If you have a book with no `author` field or if it's set to something like `None` or an empty string, this code skips over it. Ensure that this behavior is intentional.
- **Performance with large datasets**: While set comprehensions are efficient, if `books` is extremely large, generating authors this way might still introduce overhead. Always profile for performance in critical areas.
- **Non-string authors**: If the author’s name is not a string (e.g., a number), it will be included in the set too. Ensure data consistency.

By recognizing these patterns and understanding their components, you'll be better equipped to utilize similar techniques in your future coding endeavors. Happy coding!